# Preprocesamiento de los datos

Trabajo de Fin de Máster

Carlos Sanchiz Pérez

## Dependencias

In [51]:
import pandas as pd
import numpy as np

### Carga de los datos

In [52]:
# Cargamos Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [53]:
pisos_unidos = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFM/Preprocesamiento/pisos_columnas_unidas_fecha.csv", sep = ';')

/tmp/ipykernel_611/2572606768.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  pisos_unidos = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFM/Preprocesamiento/pisos_columnas_unidas_fecha.csv", sep = ';')


In [54]:
pisos_unidos.shape

(41364, 56)

## Preprocesamiento de los datos

Primero vamos a cargar nuestro conjunto de datos, y vamos a ver como se han recolectado los valores en cada columna:

In [55]:
pisos_unidos.head()

,nombre,id,detalle_url,latitud,longitud,localidad,region,precio,superficie_construida,superficie_til,...,gas,no_se_aceptan_mascotas,ascensor,planta,exterior,balcn,soleado,portero_automtico,interior,fecha
0,&#xC1;tico en Ensanche,"38313605,500626",https://www.pisos.com/comprar/atico-alcoi_alco...,"38,700241319","-0,481299659",Alcoi Alcoy,Alicante,70.000 €,90 m²,90 m²,...,NaN,NaN,NaN,5ª,TRUE,True,NaN,NaN,NaN,23/12/2025
1,"Piso en calle Dr Mara&#xF1;on, n&#xBA; 10","58089976,4433",https://www.pisos.com/comprar/piso-ibi_centro_...,"38,6241741090101","-0,574838651520533",Ibi,Alicante,139.000 €,120 m²,120 m²,...,NaN,NaN,TRUE,4ª,TRUE,True,TRUE,TRUE,NaN,24/02/2026
2,&#xC1;tico en Playa San Juan,"62379629,4342",https://www.pisos.com/comprar/atico-playas_pla...,"38,3807299","-0,4101186",Alicante Alacant,Alicante,950.000 €,100 m²,100 m²,...,NaN,NaN,TRUE,10ª,TRUE,NaN,TRUE,NaN,NaN,02/02/2026
3,Piso en Mallaeta,"70212031,2631",https://www.pisos.com/comprar/piso-montiboli_p...,"38,5024926701045","-0,241089855951997",La Vila Joiosa Villajoyosa,Alicante,200.000 €,85 m²,85 m²,...,NaN,NaN,TRUE,Bajo,TRUE,NaN,TRUE,TRUE,TRUE,24/02/2026
4,Piso en Sta.Rosa,"91701876,500626",https://www.pisos.com/comprar/piso-alcoi_alcoy...,"38,691936218","-0,488744753",Alcoi Alcoy,Alicante,115.000 €,90 m²,90 m²,...,TRUE,NaN,TRUE,Bajo,TRUE,NaN,NaN,NaN,NaN,Desconocida_Revisada


Ahora vamos a ver las dimensiones y comprobar si tienen coherencia con los resultados esperados.

In [56]:
pisos_unidos.shape

(41364, 56)

Por un lado, vemos coherente el número de filas, ya que durante la recolección el número de anuncios en ese momento era superior a 41.000. Por otro lado, observamos 55 columnas que, sin un análisis previo, no sabríamos con certeza si el número de variables explicativas tiene sentido. Primero, vamos a realizar una consulta para ver el nombre de cada una:

In [57]:
pisos_unidos.columns

Index(['nombre', 'id', 'detalle_url', 'latitud', 'longitud', 'localidad',
       'region', 'precio', 'superficie_construida', 'superficie_til',
       'superficie_solar', 'habitaciones', 'baos', 'antigedad', 'conservacin',
       'referencia', 'armarios_empotrados', 'carpintera_interior',
       'tipo_suelo', 'carpintera_exterior', 'chimenea', 'cocina_equipada',
       'comedor', 'lavadero', 'puerta_blindada', 'sistema_de_seguridad',
       'terraza', 'jardn', 'orientacin', 'urbanizado', 'calle_alumbrada',
       'calle_asfaltada', 'se_aceptan_mascotas', 'tipo_de_casa', 'garaje',
       'piscina', 'gastos_de_comunidad', 'agua', 'aire_acondicionado',
       'calefaccin', 'trastero', 'amueblado', 'vidrios_dobles', 'luz',
       'telfono', 'adaptado_a_personas_con_movilidad_reducida', 'gas',
       'no_se_aceptan_mascotas', 'ascensor', 'planta', 'exterior', 'balcn',
       'soleado', 'portero_automtico', 'interior', 'fecha'],
      dtype='object')

Observamos que todas las columnas tienen nombres con sentido. Por cómo se hizo la recolección se optó por eliminar caracteres del español, como acentos, ñ o diéresis. Como hemos visto en las primeras filas de nuestro dataframe, hay muchos valores nulos en algunas columnas, esto se debe a que la mayoría de los anunciantes no tuvieron en cuenta esa variable a la hora de crear el anuncio o simplemente como no lo tiene la vivienda no se añadió esa opción, por ejemplo si una vivienda no tenía piscina pues el anunciante no ponía "sin piscina" en las características. El primer filtro que se impuso fue que al menos la columna tuviera 5.000 datos, es decir, las columnas con menos de 5.000 datos las eliminamos:

In [58]:
pisos_unidos = pisos_unidos.dropna(axis=1, thresh=5000)
pisos_unidos.shape

(41364, 32)

Se obtuvieron 32 columnas que cumplen con dicho criterio. Analizamos ahora todas las columnas para ver cómo tratarlas o descartarlas, en caso de que no tengan valor predictivo o algún otro problema.

#### Columnas descartadas

- **nombre**: Nombre del anuncio de la vivienda. Se transformó en una nueva columna que explicaba el tipo de vivienda, llamada *tipo*.

- **id**: Número de identificación de la vivienda que estableció el portal de viviendas. A pesar de ser único, no aporta información relevante para el análisis.

- **region**: Provincia a la que pertenece la vivienda. En este caso siempre es Alicante. Tras comprobar que todas las viviendas pertenecen a Alicante, la variable se descarta por falta de variabilidad.

- **referencia**: Código interno de la plataforma. Variable no útil para el análisis.

- **superficie_util**: Superficie útil de la vivienda (en metros cuadrados). Se descartó debido a la gran cantidad de valores faltantes y porque aportaba información similar a *superficie_construida*.

- **superficie_solar**: Superficie total de la parcela (en metros cuadrados). Se descartó porque en la mayoría de los anuncios esta información era nula.

- **armarios_empotrados**: Indica la presencia de armarios empotrados. Se descartó debido a la elevada cantidad de datos faltantes y a la alta variabilidad de respuestas, lo que dificultaba su agrupación coherente.

- **cocina_equipada**: Indica la presencia de cocina equipada. Se descartó por la escasa cantidad de datos y la elevada heterogeneidad de respuestas.

- **puerta_blindada**: Indica la presencia de puerta blindada. Se descartó por la escasez de datos y la dificultad para agrupar las distintas respuestas.

- **agua**: Indica si el inmueble dispone de suministro de agua. Se descartó porque la mayoría de anuncios no informaban de esta característica, pese a que probablemente estuviera presente.

- **calefaccion**: Tipo de sistema de calefacción. Se descartó debido a la amplia variabilidad de respuestas, que no permitía una agrupación razonable en pocas categorías.

- **amueblado**: Indica si la vivienda está amueblada. Se descartó porque en muchos anuncios no se mencionaba esta característica, aún cuando la vivienda estaba amueblada.

Ahora lo hacemos en código:

In [59]:
# Crear la columna 'tipo' con la primera palabra de 'nombre'
pisos_unidos["tipo"] = (
    pisos_unidos["nombre"]
    .astype(str)                # asegurar tipo string
    .str.strip()                # eliminar espacios iniciales/finales
    .str.split()                # dividir por espacios
    .str[0]                     # quedarse con la primera palabra
)

# Lista de columnas a eliminar
columnas_a_eliminar = [
    "nombre",
    "id",
    "region",
    "referencia",
    "superficie_til",
    "superficie_solar",
    "armarios_empotrados",
    "cocina_equipada",
    "puerta_blindada",
    "agua",
    "calefaccin",
    "amueblado"
]

# Eliminar columnas (solo si existen, por seguridad)
pisos_unidos = pisos_unidos.drop(columns=[col for col in columnas_a_eliminar if col in pisos_unidos.columns])

# Comprobación
print("Columnas actuales:")
print(pisos_unidos.columns)

Columnas actuales:
Index(['detalle_url', 'latitud', 'longitud', 'localidad', 'precio',
       'superficie_construida', 'habitaciones', 'baos', 'antigedad',
       'conservacin', 'terraza', 'jardn', 'orientacin', 'garaje', 'piscina',
       'aire_acondicionado', 'trastero', 'ascensor', 'planta', 'fecha',
       'tipo'],
      dtype='object')


#### Preprocesamiento de las variables

Veamos una descripción de cada una de las columnas selecionadas y como se procesó:

- **tipo**: Tipo de la vivienda. Puede ser Apartamento, Ático, Casa, Chalet, Dúplex, Estudio, Finca, Loft y Piso. La limpieza en esta columna consistió en reemplazar los códigos HTML acentuados por su equivalente en UTF-8. Se convirtió esta variable en categórica (factor).

In [60]:
# Reemplazar códigos HTML por caracteres reales
pisos_unidos["tipo"] = pisos_unidos["tipo"].str.replace("&#xC1;", "Á", regex=False)
pisos_unidos["tipo"] = pisos_unidos["tipo"].str.replace("&#xFA;", "ú", regex=False)

# Convertir a variable categórica con orden definido (equivalente a factor(levels=...))
orden_tipos = [
    "Apartamento",
    "Ático",
    "Casa",
    "Chalet",
    "Dúplex",
    "Estudio",
    "Finca",
    "Loft",
    "Piso"
]

pisos_unidos["tipo"] = pd.Categorical(
    pisos_unidos["tipo"],
    categories=orden_tipos,
    ordered=False
)

# Tabla de frecuencias (equivalente a table())
pisos_unidos["tipo"].value_counts(dropna=False)

,count
tipo,
Chalet,11348
Piso,10770
Casa,10218
Apartamento,5712
Ático,2114
Finca,631
Dúplex,404
Estudio,147
Loft,20


- **detalle_url**: URL del anuncio. Contiene toda la información del anuncio. Es única y puede ser útil para recopilar información faltante o comprobar características, pero no resulta relevante para el estudio. La dejamos en el dataset por si necesitamos hacer alguna consulta más adelante.

- **latitud**: Coordenada geográfica que indica la posición geográfica norte-sur de la vivienda. Se expresa en grados decimales. Se recolectaron todos los datos en esta columna.

In [61]:
pisos_unidos["latitud"] = pisos_unidos["latitud"].str.replace(",", ".").astype(float)

- **longitud**: Coordenada geográfica que indica la posición geográfica este-oeste de la vivienda. Se expresa en grados decimales. Se recolectaron todos los datos en esta columna.

In [62]:
pisos_unidos["longitud"] = pisos_unidos["longitud"].str.replace(",", ".").astype(float)

- **localidad**: Municipio en el que se encuentra la vivienda. Se recolectaron datos de 137 localidades de las 141 que existen en la región de Alicante. De las localidades Alcoleja, Famorca, Fageca y Benifallim no hubo ninguna vivienda en venta en el momento de hacer la recolección, posiblemente debido a la poca población en dichos municipios. En esta columna también se reemplazó los códigos HTML acentuados por su equivalente en UTF-8. Se convirtió esta variable en categórica (factor).

In [63]:
# Reemplazar códigos HTML y corregir nombres
reemplazos = {
    "&#xED;": "í",
    "&#xE9;": "é",
    "&#xE0;": "à",
    "&#xE1;": "á",
    "&#x27;": "'",
    "&#xF2;": "ò",
    "&#xF3;": "ó",
    "&#xF1;": "ñ",
    "La Torre de les Ma&#xE7;anes Torremanzanas": "Torremanzanas",
    "&#xFA;": "ú",
    "&#xFC;": "ü"
}

for codigo, letra in reemplazos.items():
    pisos_unidos["localidad"] = pisos_unidos["localidad"].str.replace(codigo, letra, regex=False)

pisos_unidos["localidad"] = pd.Categorical(
    pisos_unidos["localidad"],
    categories=pisos_unidos["localidad"].unique(),
    ordered=False
)

# Tabla de frecuencias
pisos_unidos["localidad"].value_counts(dropna=False)

,count
localidad,
Torrevieja,5551
Orihuela,3088
Alicante Alacant,2200
Pilar de la Horadada,2161
Finestrat,1847
...,...
La Vall D'alcalà,1
L'orxa Lorcha,1
Almudaina,1


- **precio**: Precio de la vivienda. Se trata de nuestra variable a predecir. Se expresa en euros y se recolectaron todos los datos de esta columna. Los datos se recolectaron como caracteres y acompañados de €, con el siguiente formato: *XXX.XXX €*, se estandarizó para que se refleje el número en clase numérica únicamente.

In [64]:
# Eliminar puntos y el símbolo de euro, luego convertir a número
pisos_unidos["precio"] = (
    pisos_unidos["precio"]
    .str.replace(r"[\. €]", "", regex=True)  # elimina '.', espacio y '€'
    .astype(int)                           # convierte a número
)

pisos_unidos["precio"].describe()

,precio
count,4.136400e+04
mean,5.160818e+05
std,5.341472e+05
min,1.683000e+03
25%,2.250000e+05
50%,3.500000e+05
75%,5.750000e+05
max,4.400000e+06


Se observa que la vivienda más barata cuesta 1.600 € y la más cara 4.400.000 €. Este tipo de valores extremos se tratarán en la sección de Outliers, de todas maneras podemos echar un vistazo a dichas casas, empecemos por la más barata:

In [65]:
pisos_unidos[pisos_unidos["precio"] == pisos_unidos["precio"].min()]["detalle_url"].iloc[0]

'https://www.pisos.com/comprar/casa-elda_centro_urbano-45034619229_107000/'

Observando el anuncio vemos que ahora esta incluso más barato que cuando se recolectó, costando ahora 1.470 €. Parece que es una pequeña casa que se necesita reforma. Veamos la más cara:

In [66]:
pisos_unidos[pisos_unidos["precio"] == pisos_unidos["precio"].max()]["detalle_url"].iloc[0]

'https://www.pisos.com/comprar/chalet-teulada-59261691916_106600/'

En este caso, el anuncio ha sido retirado de la web, posiblemente debido a que ya ha sido comprada. Al contrario del precio más bajo, este valor no nos hacia sospechar de un error de recolección.

- **superficie_construida**: Superficie total de la vivienda. Se mide en metros cuadrados. Faltan 1551 datos. Dado que si en el anuncio de una vivienda no se nos proporciona información sobre la superficie construida no hay posibilidad de extraer dicha información, se optó por eliminar las filas que no tenían dicha información (Se hace más adelante junto con la columna habitaciones y baños).

In [67]:
pisos_unidos["superficie_construida"] = (
    pisos_unidos["superficie_construida"]
    .astype(str)                             # asegurar tipo string
    .str.replace(r"[^0-9]", "", regex=True)  # eliminar todo lo que no sea dígito
)

# convertir a número de forma segura
pisos_unidos["superficie_construida"] = pd.to_numeric(
    pisos_unidos["superficie_construida"],
    errors="coerce"  # convierte valores no válidos a NaN
)

pisos_unidos["superficie_construida"].describe()

,superficie_construida
count,3.981300e+04
mean,1.670103e+04
std,3.262486e+06
min,1.000000e+00
25%,8.500000e+01
50%,1.200000e+02
75%,2.100000e+02
max,6.509609e+08


Se observa que la vivienda más pequeña tiene 1 $m^2$ y la más grande más de 600.000.000, estas viviendas serán tratadas más adelante en la sección de Outliers.

- **habitaciones**: Número de habitaciones que dispone la vivienda. Faltan 793 datos. Se revisó porque faltaban datos y se llegó a la conclusión de que en los anuncios a veces se da por hecho que un estudio o loft tienen solamente una habitación o una finca no tiene, por ello a la hora de poner el anuncio ese valor no se tiene en cuenta y después a la hora de recogerlo aparece como NA. Se reemplazaron por dichos valores y se redujo el número de datos faltantes a 531. Las filas que aún no poseían ningún valor o valían cero, se eliminaron.

In [68]:
# Reemplazar NA o "" según tipo
pisos_unidos["habitaciones"] = np.where(
    (pisos_unidos["tipo"] == "Estudio") & ((pisos_unidos["habitaciones"].isna()) | (pisos_unidos["habitaciones"] == "")),
    1,
    np.where(
        (pisos_unidos["tipo"] == "Finca") & ((pisos_unidos["habitaciones"].isna()) | (pisos_unidos["habitaciones"] == "")),
        0,
        np.where(
            (pisos_unidos["tipo"] == "Loft") & ((pisos_unidos["habitaciones"].isna()) | (pisos_unidos["habitaciones"] == "")),
            1,
            pd.to_numeric(pisos_unidos["habitaciones"], errors="coerce")
        )
    )
)

# Ver posibles outliers iguales a 1.998
pisos_unidos.loc[pisos_unidos["habitaciones"] == 1.998, ["tipo", "habitaciones"]]

# Eliminar fila con 1.998 habitaciones
pisos_unidos = pisos_unidos[(pisos_unidos["habitaciones"].isna()) | (pisos_unidos["habitaciones"] != 1.998)]

# Ver top 10 habitaciones más altas
pisos_unidos.loc[pisos_unidos["habitaciones"].notna(), ["tipo", "habitaciones"]] \
    .sort_values(by="habitaciones", ascending=False) \
    .head(10)

# Corregir valor 45 por 5
pisos_unidos["habitaciones"] = np.where(
    pisos_unidos["habitaciones"] == 45,
    5,
    pisos_unidos["habitaciones"]
)

pisos_unidos["habitaciones"].head()

,habitaciones
0,2.0
1,4.0
2,3.0
3,3.0
4,4.0


- **banyos**: Número de baños que dispone la vivienda. Faltan 785 datos. Las filas con datos faltantes se eliminaron. Eliminamos las filas que tienen algún nulo en superficie construida, habitaciones o baños. Se optó por poner "ny" en vez de "ñ" por evitar posibles errores futuros.

In [69]:
# Filas antes de filtrar
print("Filas antes:", pisos_unidos.shape[0])

# Filtrar filas donde superficie_construida, banyos y habitaciones no sean NA ni ""
pisos_unidos = pisos_unidos[
    pisos_unidos["superficie_construida"].notna() & (pisos_unidos["superficie_construida"] != "") &
    pisos_unidos["baos"].notna() & (pisos_unidos["baos"] != "") &
    pisos_unidos["habitaciones"].notna() & (pisos_unidos["habitaciones"] != "")
]

# Ponemos el nombre bien.
pisos_unidos = pisos_unidos.rename(columns={"baos": "banyos"})

# Cambiamos un valor que se habia recogido mal:
pisos_unidos.loc[pisos_unidos["banyos"] == 23, "banyos"] = 2

# Filas después de filtrar
print("Filas después:", pisos_unidos.shape[0])

Filas antes: 41364
Filas después: 39036


- **antiguedad**: Antigüedad de la vivienda, expresada en un intervalo de tiempo, en años. Se trató como factor donde se obtuvo los siguientes niveles: "Menos de 5 años", "Entre 5 y 10 años", "Entre 10 y 20 años", "Entre 20 y 30 años", "Entre 30 y 50 años", "Más de 50 años". Por último, en los anuncios que no informaban sobre la antigüedad de esa vivienda se les atribuyó el nivel de "Desconocida". Se optó por poner "u" en vez de "ü" por evitar posibles errores futuros.

In [70]:
# Reemplazar cadenas vacías y NaN por "Desconocida"
pisos_unidos["antigedad"] = pisos_unidos["antigedad"].replace("", "Desconocida")
pisos_unidos["antigedad"] = pisos_unidos["antigedad"].fillna("Desconocida")

# Convertir a categoría con orden lógico y "Desconocida" al final
niveles_antiguedad = [
    "Menos de 5 años",
    "Entre 5 y 10 años",
    "Entre 10 y 20 años",
    "Entre 20 y 30 años",
    "Entre 30 y 50 años",
    "Más de 50 años",
    "Desconocida"
]

pisos_unidos["antigedad"] = pd.Categorical(
    pisos_unidos["antigedad"],
    categories=niveles_antiguedad,
    ordered=False
)

# Ponemos el nombre bien.
pisos_unidos = pisos_unidos.rename(columns={"antigedad": "antiguedad"})

# Tabla de frecuencias
pisos_unidos["antiguedad"].value_counts(dropna=False)

,count
antiguedad,
Desconocida,23730
Menos de 5 años,5221
Entre 30 y 50 años,3002
Entre 20 y 30 años,2584
Más de 50 años,2275
Entre 10 y 20 años,1744
Entre 5 y 10 años,480


- **conservacion**: Estado de conservación de la vivienda. Se trató como factor y se obtuvieron los siguientes niveles: "A estrenar", "A reformar", "En buen estado", "Reformado". Para los anuncios que no mencionaban en que nivel de conservación se encontraban se le atribuyó el nivel de "Desconocida". Se optó por no poner el acento para evitar un posible error en el futuro.

In [71]:
# Sustituir "" y NaN por "Desconocida"
pisos_unidos["conservacin"] = pisos_unidos["conservacin"].replace("", "Desconocida")
pisos_unidos["conservacin"] = pisos_unidos["conservacin"].fillna("Desconocida")

# Convertir a categoría con orden lógico
niveles_conservacion = [
    "A estrenar",
    "A reformar",
    "En buen estado",
    "Reformado",
    "Desconocida"
]

pisos_unidos["conservacin"] = pd.Categorical(
    pisos_unidos["conservacin"],
    categories=niveles_conservacion,
    ordered=True
)

# Ponemos el nombre bien.
pisos_unidos = pisos_unidos.rename(columns={"conservacin": "conservacion"})

# Tabla de frecuencias
pisos_unidos["conservacion"].value_counts(dropna=False)

,count
conservacion,
Desconocida,32316
En buen estado,5003
A estrenar,975
A reformar,390
Reformado,352


- **terraza**: Indica si el inmueble dispone de terraza. Se trató como variable dicotómica de la siguiente manera, si en el anuncio existía la característica "terraza", se le asigna *True* y en otro caso, se le asigna *False*.

In [72]:
print("Porcentaje de NA:", pisos_unidos["terraza"].isna().mean() * 100)

# Convertir terraza a True si hay valor, False si es NA
pisos_unidos["terraza"] = pisos_unidos["terraza"].notna()


print("Porcentaje de False:", 100 - pisos_unidos["terraza"].mean() * 100)


Porcentaje de NA: 57.544318065375556
Porcentaje de False: 57.54431806537555


- **jardin**: Indica si el inmueble dispone de jardín. Se trató como factor y se dividió en los niveles "comunitario" y "Privado". Para las viviendas donde el anuncio no mencionaba nada sobre el jardín, se le atribuyó el nivel de "Desconocido". Se optó por quitarle el acento para evitar errores en el futuro.

In [73]:
# Limpiar espacios y convertir a minusculas
jardin_limpio = pisos_unidos["jardn"].astype(str).str.strip().str.lower()

# Aplicar la lógica
pisos_unidos["jardn"] = np.where(
    (pisos_unidos["jardn"].isna()) | (pisos_unidos["jardn"].str.strip() == ""),
    "Desconocido",
    np.where(
        jardin_limpio == "comunitario",
        "Comunitario",
        "Privado"
    )
)

pisos_unidos["jardn"] = pd.Categorical(
    pisos_unidos["jardn"],
    categories=pisos_unidos["jardn"].unique(),
    ordered=False
)

# Ponemos el nombre bien.
pisos_unidos = pisos_unidos.rename(columns={"jardn": "jardin"})

# Tabla de frecuencias
pisos_unidos["jardin"].value_counts(dropna=False)

,count
jardin,
Desconocido,28000
Privado,10050
Comunitario,986


- **orientacion**: Orientación predominante de la vivienda. Se trató como factor y se dividió en los niveles "Desconocido", "Este", "Norte", "Norte-Este", "Norte-Oeste", "Oeste", "Sur", "Sur-Este", "Sur-Oeste" y "Todos". Se quitó el acento para evitar problemas en el futuro.

In [74]:
# Limpiar espacios y pasar a minúsculas
orientacion_limpia = pisos_unidos["orientacin"].astype(str).str.strip().str.lower()

# Condición inicial: NA o vacío -> NA (después lo pondremos "Desconocido")
pisos_unidos["orientacin"] = np.where(
    (pisos_unidos["orientacin"].isna()) | (pisos_unidos["orientacin"] == ""),
    np.nan,
    pisos_unidos["orientacin"]
)

# Valores que contienen "todas"/"todos" o combinaciones norte-sur o este-oeste -> "Todos"
pisos_unidos["orientacin"] = np.where(
    (orientacion_limpia.isin(["todas", "todos"])) |
    ((orientacion_limpia.str.contains("norte")) & (orientacion_limpia.str.contains("sur"))) |
    ((orientacion_limpia.str.contains("este")) & (orientacion_limpia.str.contains("oeste"))),
    "Todos",
    pisos_unidos["orientacin"]
)

# Agrupar "Norte-Este"
norte_este = [
    "Nordeste", "Noreste", "Norte este", "Norte, este", "Norte::este",
    "Norte|este", "North/east", "Norte-Este"
]
pisos_unidos.loc[pisos_unidos["orientacin"].isin(norte_este), "orientacin"] = "Norte-Este"

# Agrupar "Este"
este = ["E", "Este"]
pisos_unidos.loc[pisos_unidos["orientacin"].isin(este), "orientacin"] = "Este"

# Agrupar "Sur-Este"
sur_este = [
    "Se", "South-east", "South/east", "Sudeste", "Sur este", "Sur, este", "Sur::este",
    "Sur|este", "Sureste", "Al mar", "Vistas al mar", "Sur-Este"
]
pisos_unidos.loc[pisos_unidos["orientacin"].isin(sur_este), "orientacin"] = "Sur-Este"

# Agrupar "Sur-Oeste"
sur_oeste = ["So", "South/west", "Suroreste", "Sur-Oeste"]
pisos_unidos.loc[pisos_unidos["orientacin"].isin(sur_oeste), "orientacin"] = "Sur-Oeste"

# Valores que apuntan a "Todos"
todos_val = ["4 vientos", "Por todos lados", "Saliente-poniente-norte", "Varias", "Todos"]
pisos_unidos.loc[pisos_unidos["orientacin"].isin(todos_val), "orientacin"] = "Todos"

# Valores que aportan poca información -> "Desconocido"
desconocido_val = ["3.010", "A la montaña", "Mediodia", "TRUE", "Desconocido"]
pisos_unidos.loc[pisos_unidos["orientacin"].isin(desconocido_val), "orientacin"] = "Desconocido"

# Cambios puntuales de nombres
pisos_unidos.loc[pisos_unidos["orientacin"] == "No", "orientacin"] = "Norte-Oeste"
pisos_unidos.loc[pisos_unidos["orientacin"] == "O", "orientacin"] = "Oeste"

# Finalmente, NA o "" -> "Desconocido"
pisos_unidos.loc[
    pisos_unidos["orientacin"].isna() | (pisos_unidos["orientacin"] == ""),
    "orientacin"
] = "Desconocido"

pisos_unidos["orientacin"] = pd.Categorical(
    pisos_unidos["orientacin"],
    categories=pisos_unidos["orientacin"].unique(),
    ordered=False
)

# Ponemos el nombre bien.
pisos_unidos = pisos_unidos.rename(columns={"orientacin": "orientacion"})

# Tabla de frecuencias
pisos_unidos["orientacion"].value_counts(dropna=False)

,count
orientacion,
Desconocido,30323
Sur,3277
Sur-Este,1610
Este,1516
Todos,1339
Norte,663
Norte-Este,293
Sur-Oeste,10
Oeste,4


- **garaje**: Indica si el inmueble dispone de garaje. Se trató como factor y se dividió en "Comunitario", "Desconocido", "Opcional" y "Privado".

In [75]:
# Limpiar espacios y pasar a minúsculas
garaje_limpio = pisos_unidos["garaje"].astype(str).str.strip().str.lower()

# Aplicar la lógica
pisos_unidos["garaje"] = np.where(
    (pisos_unidos["garaje"].isna()) | (pisos_unidos["garaje"] == ""),
    "Desconocido",
    np.where(
        garaje_limpio.str.contains("comunit"),
        "Comunitario",
        np.where(
            garaje_limpio.str.contains("opcion|alquil|€|precio|mes"),
            "Opcional",
            "Privado"
        )
    )
)

pisos_unidos["garaje"] = pd.Categorical(
    pisos_unidos["garaje"],
    categories=pisos_unidos["garaje"].unique(),
    ordered=False
)

# Tabla de frecuencias
pisos_unidos["garaje"].value_counts(dropna=False)

,count
garaje,
Desconocido,30686
Privado,8269
Opcional,65
Comunitario,16


- **piscina**: Señala la presencia de piscina. Se trató como factor y nos quedaron los siguientes niveles: "Climatizada", "Comunitaria", "Exterior", "Sin piscina". Este fue el único caso donde se decidió que cuando en el anuncio no se añadía la característica de "piscina" era porque realmente no disponía la vivienda de piscina, por lo que se puso "Sin piscina", en vez de "Desconocida".

In [76]:
# Exterior
exterior = [
    "TRUE", "Yes", "∞", "Private", "Propia", "Propia, heated pool",
    "Propia, saltwater swimming pool", "Pool_with_jacuzzi", "Piscina privada",
    "Grande", "Cubierta", "Cover", "Con spa jacuzzi", "Con jaccuzzi",
    "Con bomba de calor", "8x4", "8 x 4", "5 x10", "20x8",
    "2 piscinas, una de ellas tipo playa", "Privada", "Privada exterior", "Exterior"
]
pisos_unidos.loc[pisos_unidos["piscina"].isin(exterior), "piscina"] = "Exterior"

# Climatizada
climatizada = [
    "Propia, indoor pool, saltwater swimming pool", "Piscina climatizada",
    "Interior", "Climatized", "Climatizada", "Privada climatizada", "Privada climatizada", "Climatizada"
]
pisos_unidos.loc[pisos_unidos["piscina"].isin(climatizada), "piscina"] = "Climatizada"

# Comunitaria
comunitaria = [
    "Comunitaria", "Communal", "Community", "3 piscinas comunitarias una de ellas con tobogan",
    "Comunitaria, indoor pool, heated pool", "Comunitaria"
]
pisos_unidos.loc[pisos_unidos["piscina"].isin(comunitaria), "piscina"] = "Comunitaria"

# Sin piscina
sin_piscina = [
    "", "-", "Balsa de recogida de aguas de lluvia", "Balsa propia", "Not_available",
    "Opcional", "Otro", "Piscina en urbanizacion al lado, 200€/temporada", "Piscinable.",
    "Pista de tenis", "Posibilidad de construir piscina privada", "Proyecto futuro",
    "Sin especificar", "Solo para 4 viviendas", "Sin piscina"
]
pisos_unidos.loc[pisos_unidos["piscina"].isin(sin_piscina), "piscina"] = "Sin piscina"

# Convertir NaN en "Sin piscina"
pisos_unidos["piscina"] = pisos_unidos["piscina"].fillna("Sin piscina")

pisos_unidos["piscina"] = pd.Categorical(
    pisos_unidos["piscina"],
    categories=pisos_unidos["piscina"].unique(),
    ordered=False
)

# Tabla de frecuencias
pisos_unidos["piscina"].value_counts(dropna=False)

,count
piscina,
Exterior,17674
Sin piscina,16064
Comunitaria,5229
Climatizada,69


- **aire_acondicionado**: Indica la presencia de sistema de aire acondicionado. Se trató como variable dicotómica. Se decidió poner *True* cuando el anuncio mencionara que disponía de aire acondicionado y *False* en otro caso.

In [77]:
print("Porcentaje de NaN antes:", pisos_unidos["aire_acondicionado"].isna().mean() * 100)

# Valores que consideramos False
false_vals = [
    "Ventiladores", "Sin maquina", "Not available", "Estufa", "", False, None, np.nan
]

# Convertir esos valores a False
pisos_unidos["aire_acondicionado"] = pisos_unidos["aire_acondicionado"].apply(
    lambda x: False if x in false_vals or pd.isna(x) else True
)

print("Porcentaje de False después:", (pisos_unidos["aire_acondicionado"] == False).mean() * 100)

Porcentaje de NaN antes: 65.87252792294292
Porcentaje de False después: 65.94169484578339


- **trastero**: Indica si el inmueble dispone de trastero. Se trató como variable dicotómica. Se decidió poner *True* cuando el anuncio mencionara que disponía de trastero y *False* en otro caso.

In [78]:
# Porcentaje de NaN antes
print("Porcentaje de NaN antes:", pisos_unidos["trastero"].isna().mean() * 100)

# Convertir a booleano: True si hay valor, False si es "" o NaN
pisos_unidos["trastero"] = pisos_unidos["trastero"].notna() & (pisos_unidos["trastero"] != "")

# Porcentaje de False después
print("Porcentaje de False después:", (pisos_unidos["trastero"] == False).mean() * 100)

Porcentaje de NaN antes: 83.9609591146634
Porcentaje de False después: 83.9609591146634


- **ascensor**: Indica la presencia de ascensores el edificio. Se trató como variable dicotómica. Se decidió poner *True* cuando el anuncio mencionara que disponía de aire acondicionado y *False* en otro caso.

In [79]:
# Porcentaje de NaN antes
print("Porcentaje de NaN antes:", pisos_unidos["ascensor"].isna().mean() * 100)

# Valores que consideramos False
false_vals = ["", "No", "No esta en funcionamiento", False, None, np.nan]

# Asignar False si está en la lista o es NaN, True en el resto
pisos_unidos["ascensor"] = pisos_unidos["ascensor"].apply(
    lambda x: False if x in false_vals or pd.isna(x) else True
)

# Porcentaje de False después
print("Porcentaje de False después:", (pisos_unidos["ascensor"] == False).mean() * 100)

Porcentaje de NaN antes: 77.33118147351163
Porcentaje de False después: 77.33630494927759


- **planta**: Planta del edificio donde se encuentra la vivienda. Se trató como factor ya que a pesar de recoger entre las plantas 1 y 20, a su vez, se unificó los valores "Bajo", "Principal", "Entresuelo" en "0ª", y el valor "Semisótano" se modificó a "Sótano". Se tomó la decisión de a las viviendas con el tipo "Casa", "Chalet", "Finca" asignarles el nivel "0ª". Por último, las filas donde no tenían dato se les asignó el nivel "Desconocido".

In [80]:
# Normalizar vacíos
pisos_unidos["planta"] = (
    pisos_unidos["planta"]
    .replace("", "Desconocida")
    .fillna("Desconocida")
)

# Asegurar dtype categórico
pisos_unidos["planta"] = pisos_unidos["planta"].astype("category")

# Asegurar nuevas categorías
nuevas_categorias = ["0ª", "Sótano"]

pisos_unidos["planta"] = pisos_unidos["planta"].cat.add_categories(
    [cat for cat in nuevas_categorias
     if cat not in pisos_unidos["planta"].cat.categories]
)

# Reasignaciones
tipos_pb = ["Casa", "Chalet", "Finca"]

pisos_unidos.loc[
    pisos_unidos["tipo"].isin(tipos_pb),
    "planta"
] = "0ª"

equivalencias_pb = ["Bajo", "Principal", "Entresuelo"]

pisos_unidos.loc[
    pisos_unidos["planta"].isin(equivalencias_pb),
    "planta"
] = "0ª"

pisos_unidos.loc[
    pisos_unidos["planta"] == "Semisótano",
    "planta"
] = "Sótano"

# Limpiar categorías no usadas
pisos_unidos["planta"] = pisos_unidos["planta"].cat.remove_unused_categories()

# Tabla final
pisos_unidos["planta"].value_counts(dropna=False)

,count
planta,
0ª,22745
Desconocida,9057
1ª,2211
2ª,1366
3ª,1205
4ª,881
5ª,502
6ª,240
7ª,199


- **fecha**: Fecha de la última actualización del anuncio. Esta columna se recolectó durante el mes de febrero de 2026 por lo que habían anuncios que se habían retirado de la plataforma, por lo que se les asignó el valor de *NaT* (Not a Time). El formato de fecha es AAAA-MM-DD.

In [81]:
pisos_unidos["fecha"] = pd.to_datetime(
    pisos_unidos["fecha"],
    format="%d/%m/%Y",
    errors="coerce"
)

pisos_unidos["fecha"].head()

,fecha
0,2025-12-23
1,2026-02-24
2,2026-02-02
3,2026-02-24
4,NaT


Una vez tratadas todas nuestras columnas, veamos con el comando info(), que no hay ningún null y el tipo de cada columna.

In [82]:
pisos_unidos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39036 entries, 0 to 41363
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   detalle_url            39036 non-null  object        
 1   latitud                39036 non-null  float64       
 2   longitud               39036 non-null  float64       
 3   localidad              39036 non-null  category      
 4   precio                 39036 non-null  int64         
 5   superficie_construida  39036 non-null  float64       
 6   habitaciones           39036 non-null  float64       
 7   banyos                 39036 non-null  float64       
 8   antiguedad             39036 non-null  category      
 9   conservacion           39036 non-null  category      
 10  terraza                39036 non-null  bool          
 11  jardin                 39036 non-null  category      
 12  orientacion            39036 non-null  category      
 13  garaje

También, con el comando describe(), podemos ver un resumen estadísitico de nuestras variables númericas:

In [83]:
pisos_unidos.describe()

,latitud,longitud,precio,superficie_construida,habitaciones,banyos,fecha
count,39036.000000,39036.000000,3.903600e+04,3.903600e+04,39036.000000,39036.000000,25240
mean,38.322981,-0.460635,5.205940e+05,1.692124e+04,3.112025,2.320730,2025-12-14 06:48:43.359746560
min,37.567154,-72.963700,1.683000e+03,1.000000e+00,0.000000,1.000000,2021-07-16 00:00:00
25%,38.004400,-0.739567,2.290000e+05,8.500000e+01,2.000000,2.000000,2025-11-13 00:00:00
50%,38.345972,-0.649420,3.510000e+05,1.200000e+02,3.000000,2.000000,2026-01-09 00:00:00
75%,38.595752,-0.131757,5.800000e+05,2.100000e+02,4.000000,3.000000,2026-02-16 00:00:00
max,43.598900,1.017890,4.400000e+06,6.509609e+08,30.000000,18.000000,2026-02-25 00:00:00
std,0.312724,0.503079,5.370727e+05,3.294749e+06,1.347149,1.127466,NaN


Finalmente, le cambiamos el nombre a nuestro dataset por pisos, y también el orden de las columnas, para que sea más limpio trabajar con él.

In [84]:
pisos = pisos_unidos[['tipo', 'precio', 'localidad', 'latitud', 'longitud', 'superficie_construida',
       'habitaciones', 'banyos', 'antiguedad', 'conservacion', 'terraza',
       'jardin', 'orientacion', 'garaje', 'piscina', 'aire_acondicionado',
       'trastero', 'ascensor', 'planta', 'fecha', 'detalle_url']]

#### Outliers

En esta sección, vamos a eliminar valores extremos de nuestro conjunto de datos, en particular, de las columnas *superficie_construida* y *precio*. Se realizó la limpieza usando el método IQR, con factor 3, ya que nuestra intención era eliminar posibles errores de recolección, pero siendo conscientes de que era probable que valores extremos sean correctos, además, también se pasó la variable precio a escala logarítmica, ya que el intervalo de precios era mucho más amplio.

In [85]:
# --- Crear columna log_precio de forma segura ---
pisos = pisos[pisos["precio"] > 0].copy()  # copia explícita para evitar SettingWithCopyWarning
pisos.loc[:, "log_precio"] = np.log(pisos["precio"])

# --- Función IQR ---
def eliminar_outliers_iqr(df, columna, factor=3):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - factor * IQR
    limite_superior = Q3 + factor * IQR

    return df[(df[columna] >= limite_inferior) &
              (df[columna] <= limite_superior)]

# --- Filtrar sobre superficie ---
pisos_limpio = eliminar_outliers_iqr(pisos, "superficie_construida")

# --- Dataset sin outliers usando log(precio) ---
pisos_limpio_con_log = eliminar_outliers_iqr(pisos_limpio, "log_precio")
pisos_limpio_con_log = pisos_limpio_con_log.drop(columns=["log_precio"])

# --- Dataset sin outliers usando precio normal ---
pisos_limpio_sin_log = eliminar_outliers_iqr(pisos_limpio, "precio")
pisos_limpio_sin_log = pisos_limpio_sin_log.drop(columns=["log_precio"])

# --- Limpiar columna auxiliar del dataset original ---
pisos = pisos.drop(columns=["log_precio"])

# --- Resumen ---
print("Filas originales:", len(pisos))
print("Filas tras limpieza con log:", len(pisos_limpio_con_log))
print("Filas tras limpieza sin log:", len(pisos_limpio_sin_log))

pisos = pisos_limpio_con_log

Filas originales: 39036
Filas tras limpieza con log: 38017
Filas tras limpieza sin log: 36354


Se optó por ver cuántas filas se eliminaban si se le aplicaba la escala logarítmica a la variable precio y cuantas se eliminaban sin cambiar la escala. Finalmente, se decidió usar la escala logarítmica, ya que los valores que eliminaba eran más extremos. Veamos una descripción de las dos columnas nuevamente:

In [86]:
pisos["superficie_construida"].describe()

,superficie_construida
count,38017.000000
mean,158.684220
std,108.968781
min,1.000000
25%,84.000000
50%,118.000000
75%,200.000000
max,585.000000


Observamos que el valor mínimo sigue siendo 1, lo cual no tiene sentido y puede afectar a nuestro análisis, por lo que se tomó la decisión de eliminar las filas que tenían *superficie_construida* igual a 1

In [87]:
print("Número de filas con superficie_construida igual a 1: ", len(pisos[pisos["superficie_construida"] == 1]))

pisos = pisos[pisos["superficie_construida"] != 1]

print("Número de filas final: ", len(pisos))

Número de filas con superficie_construida igual a 1:  3
Número de filas final:  38014


Veamos nuevamente la descripción de dicha columna:

In [88]:
pisos["superficie_construida"].describe()

,superficie_construida
count,38014.000000
mean,158.696664
std,108.964076
min,4.000000
25%,84.000000
50%,118.000000
75%,200.000000
max,585.000000


Veamos ahora la descripción de la columna *precio*

In [89]:
pisos["precio"].describe()

,precio
count,3.801400e+04
mean,4.870137e+05
std,4.660129e+05
min,1.600000e+04
25%,2.250000e+05
50%,3.490000e+05
75%,5.550000e+05
max,4.370000e+06


Se observa que después de la limpieza de outliers, los extremos tiene más sentido, valiendo el mínimo 16.000 € y 4.370.000 € el máximo.

In [90]:
# Elimnamos dos columnas que no tienen carácter predictivo, pero nos han ayudado durante la recolección:
pisos = pisos.drop(columns=['detalle_url', 'fecha'])

In [91]:
pisos.to_csv("pisos.csv", index=False, sep = ';')

In [92]:
pisos.shape

(38014, 19)